# CredX - Step 1: Data Cleaning & Preprocessing
This notebook covers:
1. Loading the raw dataset (10,000 alternative credit records)
2. Dataset summary, info, and missing value analysis
3. Duplicate removal & datatype validation
4. Outlier analysis & 1.5 * IQR capping on monetary fields
5. Saving the processed cleaned dataset


In [ ]:
import pandas as pd
import numpy as np
import os

# Load raw dataset
DATA_PATH = "../data/raw/alternate_credit_dataset.csv"
df = pd.read_csv(DATA_PATH)
print("Raw Dataset Shape:", df.shape)
df.head()


## 1. Missing Value Analysis
Inspect missing counts and percentages across all 41 columns.


In [ ]:
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': round((df.isnull().sum() / len(df)) * 100, 2)
})
missing[missing['Missing Count'] > 0].sort_values('Missing %', ascending=False)


## 2. Duplicate Check & Removal


In [ ]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate borrower IDs:", df.duplicated(subset=['borrower_id']).sum())
df = df.drop_duplicates().drop_duplicates(subset=['borrower_id'])
print("Shape after deduplication:", df.shape)


## 3. Domain-Aware Missing Value Imputation
- UPI metrics: 0 for non-digital users
- Rental metrics: 0 for non-renters
- Same number year: median year
- E-commerce returns & prepaid ratio: 0 and median
- Psychometric survey questions: median rating


In [ ]:
# UPI and Wallet
df['upi_transactions_per_month'] = df['upi_transactions_per_month'].fillna(0.0)
df['upi_avg_transaction_amount'] = df['upi_avg_transaction_amount'].fillna(0.0)
df['upi_months_active'] = df['upi_months_active'].fillna(0.0)
df['mobile_wallet_used'] = df['mobile_wallet_used'].fillna(0.0)

# Rent
df['rent_paid_on_time_months'] = df['rent_paid_on_time_months'].fillna(0.0)
df['total_rental_months'] = df['total_rental_months'].fillna(0.0)

# Phone tenure
df['same_number_since_year'] = df['same_number_since_year'].fillna(df['same_number_since_year'].median())

# Ecomm
df['ecomm_return_rate'] = df['ecomm_return_rate'].fillna(0.0)
df['prepaid_orders_ratio'] = df['prepaid_orders_ratio'].fillna(df['prepaid_orders_ratio'].median())

# Survey questions
for q in [f'survey_q{i}' for i in range(1, 9)]:
    df[q] = df[q].fillna(df[q].median())

print("Total missing values after imputation:", df.isnull().sum().sum())


## 4. Outlier Analysis & IQR Capping
Using 1.5 * IQR capping on income, recharge, loan request, and UPI transaction amounts.


In [ ]:
outlier_cols = [
    'income_month_1', 'income_month_2', 'income_month_3',
    'income_month_4', 'income_month_5', 'income_month_6',
    'avg_monthly_recharge_amount', 'loan_amount_requested',
    'upi_avg_transaction_amount'
]

for col in outlier_cols:
    q25 = df[col].quantile(0.25)
    q75 = df[col].quantile(0.75)
    iqr = q75 - q25
    lower = max(0.0, float(q25 - 1.5 * iqr))
    upper = float(q75 + 1.5 * iqr)
    num_capped = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: {num_capped} capped to [{lower:.2f}, {upper:.2f}]")
    df[col] = df[col].clip(lower=lower, upper=upper)


## 5. Export Cleaned Dataset


In [ ]:
OUTPUT_PATH = "../data/processed/cleaned_dataset.csv"
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print("Saved cleaned dataset to:", OUTPUT_PATH)
print("Final Shape:", df.shape)
